In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import pandas as pd
from matplotlib import pyplot as plt
import numpy as np
import torch
from torch.nn.functional import conv3d
from torch.nn import functional as F


from notebooks.imports import *

from scripts.ucla_rdk import DotMotionStim as RDK_ucla
from scripts.uw_rdk import RandomDotMotion as RDK_uw
from scripts.motion_energy import get_stimulus_array_uw, get_stimulus_array_ucla, create_motion_energy_filters_from_degrees

### Config

In [3]:
from config import dir_config

devices = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {devices}")

raw_dir = Path(dir_config.data.raw)
processed_dir = Path(dir_config.data.processed)

Using device: cuda


### Load and Preprocess data

In [4]:
data = pd.read_csv(Path(processed_dir, "valid_data.csv"), index_col=None)


# convert pulse info to frame units
convert_to_frame = 1000 / 16.67
data["pulse_onset_frame"] = np.round(data.pulse_onset * convert_to_frame) + 1
data["pulse_offset_frame"] = np.round(data.pulse_onset_frame + (data.pulse_duration * convert_to_frame))
data["rt_frame"] = np.ceil(data.reaction_time * convert_to_frame) + 1

data

,agg_index,idx_session,data_source,subject_id,session_id,idx_valid_trial,idx_correction,is_correction_trial,signed_coherence,target,choice,reaction_time,outcome,reward_amount,fixation_duration,stimulus_duration,reinforcement_duration,intertrial_delay,is_pulse_trial,pulse_onset,pulse_duration,pulse_coherence,stimulus_seed,pulse_type,pulse_onset_frame,pulse_offset_frame,rt_frame
0,0,0,uw,DBM11,10_1,1,0,False,100,1,1,1.080817,1.0,1.5,0.530901,25.0,0.0,0.500000,True,0.416750,0.066680,-100,612658,-3.0,26.0,30.0,66.0
1,1,0,uw,DBM11,10_1,2,0,False,100,1,0,0.469520,0.0,NaN,0.577326,25.0,0.0,13.689771,False,0.416750,0.066680,100,463182,0.0,26.0,30.0,30.0
2,2,0,uw,DBM11,10_1,3,0,False,-100,-1,0,2.171707,1.0,1.5,0.556890,25.0,0.0,0.500000,True,0.416750,0.066680,100,165712,3.0,26.0,30.0,132.0
3,3,0,uw,DBM11,10_1,4,0,False,-100,-1,0,0.476003,1.0,1.5,0.548262,25.0,0.0,0.500000,False,0.050010,0.066680,-100,418800,-0.0,4.0,8.0,30.0
4,4,0,uw,DBM11,10_1,5,0,False,-100,-1,0,0.971870,1.0,1.5,0.561377,25.0,0.0,0.500000,False,0.233380,0.066680,-100,372882,-0.0,15.0,19.0,60.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113869,113869,297,ucla,PSUIM9,9_1,346,0,False,18,1,1,1.108688,1.0,2.6,1.000000,60.0,0.5,0.250000,True,0.029568,0.050560,100,346,1.0,3.0,6.0,68.0
113870,113870,297,ucla,PSUIM9,9_1,347,0,False,36,1,1,1.598707,1.0,2.6,1.000000,60.0,0.5,0.250000,False,NaN,NaN,0,347,0.0,NaN,NaN,97.0
113871,113871,297,ucla,PSUIM9,9_1,348,0,False,-9,-1,1,2.292042,0.0,0.0,1.000000,60.0,1.5,0.252608,False,NaN,NaN,0,348,0.0,NaN,NaN,139.0
113872,113872,297,ucla,PSUIM9,9_1,349,0,False,36,1,1,1.706793,1.0,2.6,1.000000,60.0,0.5,0.250000,True,0.240538,0.050291,100,349,2.0,15.0,18.0,104.0


### Utils

### Helper Functions

#### Generate Stimulus with Matplotlib

In [5]:
stimulus_info = data[['agg_index', 'stimulus_seed', 'signed_coherence', 'pulse_coherence', 'pulse_onset_frame', 'pulse_offset_frame', 'rt_frame', 'data_source']]

stimulus_info

,agg_index,stimulus_seed,signed_coherence,pulse_coherence,pulse_onset_frame,pulse_offset_frame,rt_frame,data_source
0,0,612658,100,-100,26.0,30.0,66.0,uw
1,1,463182,100,100,26.0,30.0,30.0,uw
2,2,165712,-100,100,26.0,30.0,132.0,uw
3,3,418800,-100,-100,4.0,8.0,30.0,uw
4,4,372882,-100,-100,15.0,19.0,60.0,uw
...,...,...,...,...,...,...,...,...
113869,113869,346,18,100,3.0,6.0,68.0,ucla
113870,113870,347,36,0,NaN,NaN,97.0,ucla
113871,113871,348,-9,0,NaN,NaN,139.0,ucla
113872,113872,349,36,100,15.0,18.0,104.0,ucla


---
---

In [ ]:
# --- CONFIGURATION ---
batch_size = 100
screen_resolution_uw = (1280, 720)
screen_resolution_ucla = (1920, 1080)
ppd_uw = 10
ppd_ucla = 20

# --- OUTPUT LIST ---
stimuli_list = []
rdk_ucla = RDK_ucla()
rdk_uw = RDK_uw()

rows = [
	{
		"agg_index": row.agg_index,
		"stimulus_seed": row.stimulus_seed,
		"pulse_onset_frame": row.pulse_onset_frame,
		"pulse_coherence": row.pulse_coherence,
		"pulse_offset_frame": row.pulse_offset_frame,
		"signed_coherence": row.signed_coherence,
		"rt_frame": row.rt_frame,
		"data_source": row.data_source,
	}
	for row in stimulus_info.itertuples(index=False)
]

rows_to_save = rows.copy()

In [ ]:
stimuli_dir = Path(processed_dir) / "stimuli_arrays"
stimulus_files = list(stimuli_dir.glob("stimulus_*.pt"))
saved_agg_indices = [f.stem.split("_")[1] for f in stimulus_files]
saved_agg_indices = [int(i) for i in saved_agg_indices]

# Convert rows to DataFrame
df = pd.DataFrame(rows)
df_to_save = df[~df["agg_index"].isin(saved_agg_indices)]
rows_to_save = df_to_save.to_dict(orient="records")

print(f"Number of stimuli to process: {len(rows_to_save)}")

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed
import torch
import os
from pathlib import Path
from threading import Lock

# --- SETUP ---
output_dir = Path(processed_dir) / "stimuli_arrays"
output_dir.mkdir(parents=True, exist_ok=True)  # Create directory if it doesn't exist
max_workers = 8
batch_size = 32  # Adjust batch size based on your memory limits

total_rows = len(rows_to_save)
indexed_rows = list(enumerate(rows_to_save))
progress_step = max(1, total_rows // 100)  # Every 1%

# Lock for thread-safe file saving and progress updates
save_lock = Lock()
progress_lock = Lock()


# --- UTILITIES ---
def batched(iterable, batch_size):
	"""Yield successive n-sized chunks from iterable."""
	for i in range(0, len(iterable), batch_size):
		yield iterable[i : i + batch_size]


def safe_int(val, default=0):
	"""Safely converts a value to int, returns default if not possible."""
	try:
		return int(val)
	except (ValueError, TypeError):
		return default


# --- WORKER FUNCTION ---
def create_stim_array(row_dict, index, rdk_ucla, rdk_uw):
	"""Creates a stimulus array based on the row data."""
	try:
		pars = {
			"seed": row_dict["stimulus_seed"],
			"coherence": row_dict["signed_coherence"],
		}

		if row_dict["pulse_onset_frame"] is not None and row_dict["pulse_offset_frame"] is not None:
			pars["pulse"] = [
				(safe_int(row_dict["pulse_onset_frame"]), row_dict["pulse_coherence"]),
				(safe_int(row_dict["pulse_offset_frame"]), row_dict["signed_coherence"]),
			]

		rt_frame = safe_int(row_dict["rt_frame"])

		if row_dict["data_source"] == "ucla":
			array, _ = get_stimulus_array_ucla(screen_resolution_ucla, rdk_ucla, pars, rt_frame, ppd=ppd_ucla, convert_to_degrees=True, return_coords=True)
		else:
			array, _ = get_stimulus_array_uw(screen_resolution_uw, rdk_uw, pars, rt_frame, ppd=ppd_uw, convert_to_degrees=True, return_coords=True)

		return array

	except Exception as e:
		print(f"[{index}] Error in processing: {e}")
		return None


# --- PARALLEL SAVE FUNCTION ---
def process_and_save(row_dict):
	"""Processes and saves stimulus arrays to disk."""
	agg_index = safe_int(row_dict.get("agg_index", -11111))  # Fallback to 0 if agg_index is missing
	array = create_stim_array(row_dict, agg_index, rdk_ucla, rdk_uw)
	if array is not None:
		try:
			tensor = torch.from_numpy(array)
			save_path = output_dir / f"stimulus_{agg_index:05d}.pt"

			# Locking the saving operation to avoid race conditions
			with save_lock:
				torch.save(tensor, save_path)

			# Locking the progress update to ensure it’s thread-safe
			with progress_lock:
				return agg_index, True
		except Exception as e:
			print(f"[{agg_index}] Save error: {e}")
	return agg_index, False


# --- EXECUTION ---
processed_count = 0
total_batches = len(rows_to_save) // batch_size + int(len(rows_to_save) % batch_size != 0)

# Process each batch of rows
for batch_index, batch in enumerate(batched(rows_to_save, batch_size)):
	print(f"\n🔁 Processing Batch {batch_index + 1}/{total_batches}...")

	with ProcessPoolExecutor(max_workers=max_workers) as executor:
		futures = [executor.submit(process_and_save, row_dict) for row_dict in batch]

		for future in as_completed(futures):
			try:
				_, success = future.result()  # Wait for the result to ensure it completes
				if success:
					with progress_lock:
						processed_count += 1  # Increment when saved successfully
			except Exception as e:
				print(f"Process error: {e}")

	# Update progress every 1% (or more, depending on your progress_step)
	if batch_index % progress_step == 0 or batch_index == total_batches:
		with progress_lock:
			percent = int((batch_index / total_batches) * 100)
			print(f"✅ Progress: {percent}% ({batch_index + 1}/{total_batches}) | ✔ Saved: {processed_count}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML


def random_stimulus_check(agg_index, stimuli_dir):
	"""Check if a stimulus file exists for the given agg_index."""
	stimulus_file = stimuli_dir / f"stimulus_{agg_index:05d}.pt"
	if not stimulus_file.exists():
		print(f"Stimulus file {stimulus_file} does not exist.")
		return False

	stimulus_array = torch.load(stimulus_file)

	stimulus_array = stimulus_array.numpy()  # Convert to numpy array for processing

	# print row
	row = stimulus_info[stimulus_info["agg_index"] == agg_index].iloc[0]
	print(f"Row for agg_index {agg_index}: {row.to_dict()}")

	# Set up the figure and axis
	fig, ax = plt.subplots()
	im = ax.imshow(stimulus_array[0], cmap="gray", vmin=stimulus_array.min(), vmax=stimulus_array.max())
	ax.set_title("Stimulus Animation")


	# Update function for the animation
	def update(frame):
		im.set_array(stimulus_array[frame])
		ax.set_xlabel(f"Time: {frame}")
		ax.grid(False)
		return [im]


	# Create the animation
	ani = animation.FuncAnimation(fig, update, frames=stimulus_array.shape[0], interval=50, blit=False)

	# Display inline (alternative if %matplotlib notebook doesn't work well)
	return HTML(ani.to_jshtml())


random_stimulus_check(agg_index=113000, stimuli_dir=stimuli_dir)  # Change agg_index to test different stimuli

---
---

## Filters

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Define the pixels per degree for mouse visual resolution
ppd = 10  # Lower PPD for mouse (in contrast to human-level resolution)

# Define the stimulus range in degrees
x_range_deg = (-6, 6)
y_range_deg = (-6, 6)

# Define other parameters as needed
dx = 1.0 / ppd  # Step size in pixels per degree
dy = 1.0 / ppd  # Step size in pixels per degree
dt = 1 / 60  # Time step, 60 FPS for example
t_range_s = (0, 0.1)  # Duration of 0.1 second for temporal range

# Create the motion energy filters with adjusted resolution for mouse
right_filters, left_filters, (x, y, t) = create_motion_energy_filters_from_degrees(
	rf_size_deg=8.0,  # Large receptive field, typical for mouse V1 neurons
	surround_width_deg=4.0,  # Surround width
	temporal_tuning_hz=5.0,  # Temporal tuning for slow motion
	x_range_deg=x_range_deg,
	y_range_deg=y_range_deg,
	t_range_s=t_range_s,
	dx=dx,
	dy=dy,
	dt=dt,
	device=device,
)

right_filters_tensor = [f.unsqueeze(0).unsqueeze(0) for f in right_filters]  # [1, 1, D, H, W]
left_filters_tensor = [f.unsqueeze(0).unsqueeze(0) for f in left_filters]
all_filters = right_filters_tensor[:2] + left_filters_tensor[:2]  # List of 4 tensors
filters_tensor = torch.cat(all_filters, dim=0).to(device)  # Final shape: [4, 1, D, H, W]


In [7]:
def calculate_ME(stimulus_array, filters_tensor, device="cuda"):
	stimulus_tensor = stimulus_array.unsqueeze(0).unsqueeze(0).float().to(device)
	_, _, kD, kH, kW = filters_tensor.shape
	pad = (kW // 2, kW // 2, kH // 2, kH // 2, kD // 2, kD // 2)
	stimulus_padded = F.pad(stimulus_tensor, pad)
	conv_out = F.conv3d(stimulus_padded, filters_tensor, padding=0)
	conv_out_sq = conv_out**2
	ME = conv_out_sq[0, 0] + conv_out_sq[0, 1] - conv_out_sq[0, 2] - conv_out_sq[0, 3]
	return ME

In [12]:
stimuli_dir = Path(processed_dir) / "stimuli_arrays"
stimulus_files = sorted(stimuli_dir.glob("stimulus_*.pt"))

output_dir = Path(processed_dir) / "motion_energy_arrays"
output_dir.mkdir(parents=True, exist_ok=True)

motion_energy_saved = (output_dir.glob("me_array*.pt"))
motion_energy_saved_indices = [int(f.stem.split("_")[2]) for f in motion_energy_saved]


stimulus_files_indices = [int(f.stem.split("_")[1]) for f in stimulus_files]
stimulus_files_to_process = [f for f in stimulus_files if int(f.stem.split("_")[1]) not in motion_energy_saved_indices]
assert len(stimulus_files) == len(stimulus_files_to_process) + len(motion_energy_saved_indices)
print(f"Processing {len(stimulus_files_to_process)} stimuli...")

Processing 50596 stimuli...


In [13]:
import torch
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

# Custom Dataset class
class StimulusDataset:
    def __init__(self, stimulus_files, filters_tensor, device):
        self.stimulus_files = stimulus_files
        self.filters_tensor = filters_tensor
        self.device = device

    def __len__(self):
        return len(self.stimulus_files)

    def __getitem__(self, idx):
        stimulus_file = self.stimulus_files[idx]
        agg_index = int(stimulus_file.stem.split("_")[1])
        try:
            stimulus_array = torch.load(stimulus_file).float().to(self.device)
        except Exception as e:
            print(f"Error loading {stimulus_file}: {e}")
            return None, None
        ME = calculate_ME(stimulus_array, self.filters_tensor)
        return agg_index, ME

# Save function
def save_motion_energy(agg_index, me_array, output_dir):
    save_path = output_dir / f"me_array_{agg_index:05d}.pt"
    torch.save(me_array, save_path)

# Example usage without DataLoader or batching
dataset = StimulusDataset(stimulus_files_to_process, filters_tensor, device)

with ThreadPoolExecutor(max_workers=8) as executor:
    futures = []
    completed = 0  # Progress counter

    # Iterate over individual samples
    for i in range(len(dataset)):
        agg_index, me_array = dataset[i]
        if agg_index is None or me_array is None:
            continue
        me_array = me_array.cpu()
        futures.append(executor.submit(save_motion_energy, agg_index, me_array, output_dir))

    for future in as_completed(futures):
        future.result()
        completed += 1
        if completed % 100 == 0:
            print(f"Processed {completed} files...")


Processed 100 files...
Processed 200 files...
Processed 300 files...
Processed 400 files...
Processed 500 files...
Processed 600 files...
Processed 700 files...
Processed 800 files...
Processed 900 files...
Processed 1000 files...
Processed 1100 files...
Processed 1200 files...
Processed 1300 files...
Processed 1400 files...
Processed 1500 files...
Processed 1600 files...
Processed 1700 files...
Processed 1800 files...
Processed 1900 files...
Processed 2000 files...
Processed 2100 files...
Processed 2200 files...
Processed 2300 files...
Processed 2400 files...
Processed 2500 files...
Processed 2600 files...
Processed 2700 files...
Processed 2800 files...
Processed 2900 files...
Processed 3000 files...
Processed 3100 files...
Processed 3200 files...
Processed 3300 files...
Processed 3400 files...
Processed 3500 files...
Processed 3600 files...
Processed 3700 files...
Processed 3800 files...
Processed 3900 files...
Processed 4000 files...
Processed 4100 files...
Processed 4200 files...
P

In [ ]:
# free cuda memory
torch.cuda.empty_cache()

In [ ]:
for i in range(0, len(motion_energy)):
	plt.plot(motion_energy[i].mean(axis=(1, 2)))
plt.xlim(0, 50)